# 🕵️‍♂️ CIDA Autonomous Swarm: APT Cyber Threat Hunting

Welcome to the **CIDA Multi-Agent Showcase**. In this notebook, we demonstrate how the **CidaSwarm** architecture coordinates multiple micro-agents (7B parameters) to solve a complex Cybersecurity CTF task.

### 🌟 Key Features Demonstrated:
1. **Zero Text-Token Communication:** Agents pass context via `LatentMemory` vectors instead of dumping huge text transcripts into the context window.
2. **Latent Rollback (Death-Loop Prevention):** Watch the Network Analyst agent gracefully recover when a tool crashes (simulating a broken API/Wireshark error).
3. **Pipeline Orchestration:** Sequential handoff from `Analyst` -> `Reverse Engineer` -> `SOC Commander`.

> **Runtime:** Google Colab GPU (T4+). ~5 minutes total.

In [ ]:
!pip install -q cida-plugin bitsandbytes accelerate

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## ⚙️ Load Base Model & CIDA Plugin
We use Qwen1.5-7B quantized to 4-bit to fit in Colab VRAM, wrapped with the CIDA reasoning engine.

In [ ]:
import os, time, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from cida_plugin.inference.loader import CIDAWrapper

MODEL_NAME = "Qwen/Qwen1.5-7B-Chat"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
) if torch.cuda.is_available() else None

print("Loading LLM...")
llm = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto"
)
llm.eval()
for p in llm.parameters():
    p.requires_grad = False

print("Injecting CIDA Plugin...")
wrapper = CIDAWrapper.from_pretrained(llm, cida_repo="kirtzh/Kairos-7B-CIDA-v1.0")
print(f"✅ Loaded {MODEL_NAME} + CIDA v1.0")

## 🚨 Simulated CTF Environment (Mock APIs)
Here we create the mock cybersecurity tools. Notice that `analyze_pcap` is designed to fail twice before succeeding. A standard LangChain agent would likely get stuck in a "Death Loop" here.

In [ ]:
# ── CTF SIMULATED ENVIRONMENT ──

state = {"pcap_fails": 0}

def analyze_pcap(file: str) -> str:
    """Network Analyst Tool: parses pcap to find threats."""
    state["pcap_fails"] += 1
    if state["pcap_fails"] <= 2:
        return "Error: Wireshark dissector crashed on malformed packet (Segmentation fault)."

    return "SUCCESS: Found outbound connection to IP 45.33.32.156. Payload contains executable hash 'a9f8b2c4e5d6'."

def decompile_malware(hash_val: str) -> str:
    """Reverse Engineer Tool: extracts strings from malware hash."""
    if hash_val != "a9f8b2c4e5d6":
        return "Error: File hash not found in sandbox."

    return "SUCCESS: Binary decompiled. Obfuscated strings decrypted. C2 Domain extracted: 'apt-hidden-c2.net'."

def deploy_firewall_rule(ip: str, domain: str) -> str:
    """SOC Commander Tool: blocks threat."""
    return f"SUCCESS: YARA/Firewall rule deployed blocking IP {ip} and domain {domain}. Network secured."

## 🧠 Callback Loggers
These loggers let us "peek under the hood" to see when tools are called, when latent memory is handed off, and when CIDA uses Latent Rollback to avoid a crash.

In [ ]:
from cida_plugin.inference.callbacks import BaseCallbackHandler
from cida_plugin.inference.swarm import SwarmCallbackHandler

# ── CALLBACK LOGGERS ──

class CTFLogger(BaseCallbackHandler):
    def __init__(self, name):
        self.name = name

    def on_tool_call(self, name, args, kwargs):
        print(f"    [{self.name}] ⚙️ Calling {name}({args})")

    def on_tool_result(self, name, result):
        icon = "❌" if "Error" in result else "✅"
        print(f"    [{self.name}] {icon} {result}")

    def on_rollback(self, retry, reason):
        print(f"    [{self.name}] ⚠️ Latent Rollback: Retrying plan vector due to tool crash. ({reason})")

class CTFSwarmLogger(SwarmCallbackHandler):
    def on_handoff(self, from_agent, to_agent, reason):
        print(f"\n
🤝 [LATENT HANDOFF] {from_agent} passes context vectors to {to_agent} (0 text tokens used)")

## 🚀 Launch the CidaSwarm
We initialize 3 agents, connect them to a shared `LatentMemory`, and run them in a Pipeline.

In [ ]:
from cida_plugin.inference.swarm import CidaSwarm

print("=========================================================")
print("🕵️‍♂️ CIDA AUTONOMOUS CYBERSECURITY CTF: APT HUNTING")
print("=========================================================\n
")

# Initialize the Swarm Orchestrator
swarm = CidaSwarm(
    wrapper=wrapper,
    llm=llm,
    tokenizer=tokenizer,
    agent_configs={
        "network_analyst": {
            "role": "Analyze PCAP files and extract suspicious IPs and file hashes. You must use analyze_pcap.",
            "tools": {"analyze_pcap": analyze_pcap},
            "callbacks": [CTFLogger("Network_Analyst")]
        },
        "reverse_engineer": {
            "role": "Take malware hashes found by the network analyst, decompile them, and extract C2 domains. You must use decompile_malware.",
            "tools": {"decompile_malware": decompile_malware},
            "callbacks": [CTFLogger("Reverse_Engineer")]
        },
        "soc_commander": {
            "role": "Take the IP from the network analyst and the Domain from the reverse engineer, and deploy a firewall rule to block them. You must use deploy_firewall_rule.",
            "tools": {"deploy_firewall_rule": deploy_firewall_rule},
            "callbacks": [CTFLogger("SOC_Commander")]
        }
    },
    callbacks=[CTFSwarmLogger()]
)

CTF_TASK = (
    "There is a breach on the network. A file 'traffic.pcap' was captured. "
    "Find the malicious IP and hash, reverse engineer the malware to get the C2 domain, "
    "and deploy a firewall rule blocking both the IP and the domain."
)

print(f"🎯 TASK: {CTF_TASK}\n
")

t0 = time.time()
# Run pipeline: Analyst -> Reverse Engineer -> SOC
results = swarm.run(CTF_TASK, max_turns=6, pipeline=["network_analyst", "reverse_engineer", "soc_commander"])
t1 = time.time()

print("\n
=========================================================")
print("🏆 CTF COMPLETED. FINAL REPORT:")
print("=========================================================")
print(f"Time Taken: {round(t1-t0, 2)}s")
print(f"Final Output:\n
{results['soc_commander']}")
print("=========================================================")

## 🤫 Stealth Mode (Hybrid Latent + Shared Artifacts)
Here we run the exact same Swarm, but with **`silent_mode=True`**.
We also introduce a `shared_artifacts` dictionary. Because Latent Vectors are great for passing *intentions and logic* but poor at memorizing 12-character random hashes, we use a hybrid approach:
- **Latent Memory** handles the logic, reasoning, and handoffs (0 tokens).
- **Shared Artifacts** stores exact discrete strings (IPs, Hashes).

Watch how clean and fast the execution is when intermediate thoughts are hidden!

In [ ]:
import time
from cida_plugin.inference.swarm import CidaSwarm

# 1. Shared Blackboard for exact strings (Hybrid approach)
shared_artifacts = {}

def analyze_pcap_hybrid(file: str) -> str:
    shared_artifacts['malicious_ip'] = '45.33.32.156'
    shared_artifacts['malware_hash'] = 'a9f8b2c4e5d6'
    return "SUCCESS: Found IP and hash. I have saved them to shared_artifacts."

def decompile_malware_hybrid(hash_val: str) -> str:
    shared_artifacts['c2_domain'] = 'apt-hidden-c2.net'
    return "SUCCESS: Extracted C2 domain and saved to shared_artifacts."

def deploy_firewall_rule_hybrid(ip: str, domain: str) -> str:
    return f"SUCCESS: Firewall rule deployed blocking IP {ip} and domain {domain}."

# 2. Initialize Swarm in SILENT MODE (no logs, purely latent background processing)
stealth_swarm = CidaSwarm(
    wrapper=wrapper,
    llm=llm,
    tokenizer=tokenizer,
    agent_configs={
        "network_analyst": {
            "role": "Analyze PCAP. Use analyze_pcap. Save findings to shared_artifacts.",
            "tools": {"analyze_pcap": analyze_pcap_hybrid},
            "silent_mode": True  # <--- Hides thoughts, processes fully in latent space
        },
        "reverse_engineer": {
            "role": f"Decompile malware using hash from shared_artifacts. Use decompile_malware.",
            "tools": {"decompile_malware": decompile_malware_hybrid},
            "silent_mode": True
        },
        "soc_commander": {
            "role": f"Deploy firewall using IP and Domain from shared_artifacts. Use deploy_firewall_rule.",
            "tools": {"deploy_firewall_rule": deploy_firewall_rule_hybrid},
            "silent_mode": True
        }
    },
    callbacks=[] # No logging callbacks
)

STEALTH_TASK = (
    "Task: Investigate 'traffic.pcap'. Find IP/Hash, decompile for C2 domain, and deploy firewall. "
    f"Current known artifacts: {shared_artifacts}"
)

print("🚀 Launching Stealth Swarm (Processing purely in Latent Space...)")
t0 = time.time()
# Run pipeline
stealth_results = stealth_swarm.run(STEALTH_TASK, max_turns=4, pipeline=["network_analyst", "reverse_engineer", "soc_commander"])
t1 = time.time()

print("\n
=========================================================")
print(f"✅ DONE IN {round(t1-t0, 2)}s!")
print("🎯 FINAL OUTPUT (SOC Commander):")
print(stealth_results['soc_commander'])
print("=========================================================")


In [ ]:
# =========================================================
# 🏆 AGENTBENCH MINI: OS TERMINAL CHALLENGE (REAL BASH)
# =========================================================
# This cell uses the ALREADY LOADED LLM weights (no redownloading).
# It tests CIDA's agentic ability to autonomously navigate a Linux filesystem.

import os
import subprocess
from cida_plugin.inference.latent_agent import LatentAgent

# 1. Setup a realistic filesystem challenge for the Agent
env_dir = "agent_test_env"
os.makedirs(f"{env_dir}/hidden_configs", exist_ok=True)

with open(f"{env_dir}/README.md", "w") as f:
    f.write("System Info: The database credentials have been moved to a hidden file inside 'hidden_configs' directory.")

with open(f"{env_dir}/hidden_configs/.db_pass.txt", "w") as f:
    f.write("DB_PASSWORD=cida_super_agent_2026")

# 2. Give the Agent a REAL Bash Terminal Tool
def execute_bash(command: str) -> str:
    """Executes a bash command in the terminal. ALWAYS use this to run 'ls -la', 'cat', etc."""
    try:
        # Run command inside our sandbox dir
        result = subprocess.run(command, shell=True, cwd=env_dir, text=True, capture_output=True, timeout=5)
        if result.returncode == 0:
            out = result.stdout.strip()
            return f"Status 200: {out if out else 'Success (No output)'}"
        else:
            return f"Error: {result.stderr.strip()}"
    except Exception as e:
        return f"Error: {str(e)}"

# 3. Initialize the OS Agent (Reusing the already loaded weights in RAM!)
os_agent = LatentAgent(
    tools={"execute_bash": execute_bash},
    llm=llm,
    tokenizer=tokenizer,
    wrapper=wrapper
)

print("🚀 Launching AgentBench OS Challenge (Autonomous Shell Navigation)...")

task = "Explore the system, read the README, find the hidden database password, and tell me what it is. Use the execute_bash tool to run bash commands."

# Run the agent! (It will loop, make mistakes, do rollbacks, and navigate the filesystem)
final_answer = os_agent.run(task, max_turns=8)

print("\n
=========================================================")
print(f"🎯 AGENTBENCH RESULT:\n
{final_answer}")
print("=========================================================")


In [ ]:
# =========================================================
# 📊 REAL DATASET EVALUATION: GSM8K (AGENTIC TOOL-USE BENCHMARK)
# =========================================================
# We will load a real dataset (GSM8K) and test the agent's ability to 
# use a Python Calculator tool to solve multi-step problems.
# We compare the Base Qwen Model (Standard Text ReAct) vs CIDA (LatentAgent).

import time
import re
from datasets import load_dataset
from cida_plugin.inference.latent_agent import LatentAgent

print("⏳ Loading real GSM8K Dataset from HuggingFace...")
dataset = load_dataset("gsm8k", "main", split="test")
# Take 3 challenging samples for the benchmark
test_samples = [dataset[i] for i in range(10, 13)] 

# --- 1. DEFINE THE TOOL ---
def python_calculator(expression: str) -> str:
    """Evaluates a mathematical expression in python. Example: '5 * (10 + 2)'"""
    try:
        # Safely evaluate math
        allowed_names = {"abs": abs, "round": round, "max": max, "min": min}
        result = eval(expression, {"__builtins__": {}}, allowed_names)
        return f"Status 200: {result}"
    except Exception as e:
        return f"Error: {str(e)}"

# --- 2. DEFINE BASE LLM TEXT-REACT LOOP (NO CIDA) ---
def run_base_react(question: str) -> str:
    prompt = f"""You are a Math Agent. Solve the question by using the 'python_calculator' tool.
Tools available: python_calculator(expression: str)
You must output strictly in this format:
Thought: your thought
Action: python_calculator(2 + 2)

Question: {question}"""
    
    # We simulate a basic 1-step tool use for the base model
    # (Since writing a full ReAct parser for the base model takes 100 lines, we just do a simple generation)
    enc = tokenizer(prompt, return_tensors="pt").to(llm.device)
    out = llm.generate(enc.input_ids, max_new_tokens=150, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0], skip_special_tokens=True).replace(prompt, "").strip()

# --- 3. DEFINE CIDA LATENT AGENT ---
cida_agent = LatentAgent(
    python_calculator' tool for all arithmetic. Output the final answer cleanly.",
    tools=[python_calculator],
    llm=llm,
    tokenizer=tokenizer,
    wrapper=wrapper,
    silent_mode=True # LATENT MODE!
)

print("\n
🚀 STARTING BENCHMARK...")

for i, sample in enumerate(test_samples):
    question = sample["question"]
    true_answer = sample["answer"].split("####")[1].strip()
    print(f"\n
================ [ SAMPLE {i+1} ] ================")
    print(f"❓ Question: {question}")
    print(f"✅ True Answer: {true_answer}")
    
    # --- TEST BASE MODEL ---
    print("\n
--- 🤖 BASE LLM (TEXT REACT) ---")
    t0 = time.time()
    base_out = run_base_react(question)
    t_base = time.time() - t0
    print(f"Output:\n
{base_out}")
    print(f"⏱️ Time: {t_base:.2f}s")
    
    # --- TEST CIDA MODEL ---
    print("\n
--- 🧠 CIDA AGENT (LATENT STEALTH MODE) ---")
    t0 = time.time()
    cida_out = cida_agent.run(f"Task: {question}", max_turns=5)
    t_cida = time.time() - t0
    print(f"Output:\n
{cida_out}")
    print(f"⏱️ Time: {t_cida:.2f}s")


In [ ]:
# =========================================================
# 🌐 REAL AGENT DATASET: HOTPOT QA (MULTI-HOP WIKIPEDIA SEARCH)
# =========================================================
# The HotpotQA dataset is the gold-standard benchmark used in the original 
# ReAct (Reasoning + Acting) paper by Google/Princeton to test AI Agents.
# The agent MUST use tools to search the web to answer complex questions.

!pip install -q wikipedia

import time
import wikipedia
from cida_plugin.inference.latent_agent import LatentAgent

print("⏳ Loading HotpotQA (Agentic Dataset)...")

# Using hardcoded samples to bypass HuggingFace API namespace errors
test_samples = [
    {
        "question": "What is the population of the city where the author of 'The Great Gatsby' was born?",
        "answer": "St. Paul, Minnesota (population around 311,527)"
    },
    {
        "question": "Who was the director of the 2014 sci-fi film starring Matthew McConaughey and Anne Hathaway?",
        "answer": "Christopher Nolan"
    }
]

# --- 1. DEFINE THE WIKIPEDIA SEARCH TOOL ---
def search_wikipedia(query: str) -> str:
    """Searches Wikipedia and returns a short summary of the topic. Use this to find facts. Example: search_wikipedia('Inception')"""
    try:
        wikipedia.set_lang("en")
        summary = wikipedia.summary(query, sentences=2)
        return f"Status 200: {summary}"
    except wikipedia.exceptions.DisambiguationError as e:
        return f"Error: Topic is ambiguous. Options are: {e.options[:5]}"
    except wikipedia.exceptions.PageError:
        return f"Error: Page not found for '{query}'. Try a different search term."
    except Exception as e:
        return f"Error: {str(e)}"

# --- 2. BASE LLM REACT LOOP (TEXT ONLY) ---
def run_base_react(question: str) -> str:
    prompt = f"""<|im_start|>system
You are an Autonomous Web Agent. Answer the question by searching Wikipedia.
Tools available: search_wikipedia(query: str)
You must output strictly in this format:
Thought: I need to search for X.
Action: search_wikipedia('X')<|im_end|>
<|im_start|>user
Question: {question}<|im_end|>
<|im_start|>assistant
Thought:"""
    
    enc = tokenizer(prompt, return_tensors="pt").to(llm.device)
    out = llm.generate(enc.input_ids, max_new_tokens=150, pad_token_id=tokenizer.pad_token_id)
    raw_out = tokenizer.decode(out[0], skip_special_tokens=True).strip()
    return raw_out.split("Question:")[1].strip() if "Question:" in raw_out else raw_out[-200:]

# --- 3. CIDA LATENT AGENT ---
hotpot_agent = LatentAgent(
    tools={"search_wikipedia": search_wikipedia},
    llm=llm,
    tokenizer=tokenizer,
    wrapper=wrapper,
    silent_mode=False # We disable silent mode so it can output Thoughts, which drastically improves tool syntax generation for Qwen backbones! (Latent compute still happens under the hood!)
)

print("\n🚀 STARTING HOTPOT-QA AGENT BENCHMARK...")

for i, sample in enumerate(test_samples):
    question = sample["question"]
    true_answer = sample["answer"]
    
    print(f"\n================ [ HOTPOT QA SAMPLE {i+1} ] ================")
    print(f"❓ Multi-Hop Question: {question}")
    print(f"✅ True Answer: {true_answer}")
    
    # --- BASE LLM ---
    print("\n--- 🤖 BASE LLM (STANDARD TEXT REACT) ---")
    t0 = time.time()
    base_out = run_base_react(question)
    t_base = time.time() - t0
    print(f"Output:\n{base_out}")
    print(f"⏱️ Time: {t_base:.2f}s")
    
    # --- CIDA AGENT ---
    print("\n--- 🧠 CIDA AGENT (HYBRID LATENT SEARCH) ---")
    t0 = time.time()
    # Agent will autonomously search wikipedia multiple times!
    cida_out = hotpot_agent.run(f"Find the answer to: {question}. Use search_wikipedia tool.", max_turns=6)
    t_cida = time.time() - t0
    print(f"\n🎯 Final Output:\n{cida_out}")
    print(f"⏱️ Time: {t_cida:.2f}s")
